# 19 · Why a graph: the loop as state, nodes, edges and checkpoints
Level L4 · Part 4 · core · ~30 min · builds on 06, 15

## Goal
Run the same refund request twice: once as the loop from lesson 06, once as a LangGraph graph. Show that the graph
pauses for a person, keeps its place through a crash in the payment, and cannot pay the same refund twice. The loop
can do none of the three.

## Why it matters
A loop keeps its state in local variables. Its chat and its calls exist only while the process runs, and nothing
outside the process knows where the loop stopped. The cell below is the loop's refund step with the model left out,
so nothing is called yet. **Run it:** the refund is paid the moment the request appears, with no person in between.

In [1]:
MOVED = [("2207", 84.00)]  # the refund the model would ask for, typed in here: the money moves inside the loop
print("money moved before any person looked:", MOVED)

money moved before any person looked: [('2207', 84.0)]


**What it makes easy:** a loop is a few lines, and the model picks the next step. **What it costs:** if the process
dies after the payment, nothing outside the process says where the loop was. **When you don't need it:** a read-only
lookup with no pause, a fixed sequence of steps (lesson 11), or one model call.

## The idea
A graph splits the loop into parts you can name. The **state** is one typed record that every step reads and writes.
A **node** is a function that reads the state and returns only what it changed. An **edge** says which node runs
next, and a **conditional edge** picks the next node from the state, such as "a refund was asked, so go to approval".
A **checkpointer** saves the state after every node, under a **thread id**. `interrupt()` stops the run at a node and
saves it; `Command(resume=...)` continues the same thread later, in another process.

Analogy: a clinic chart. The chart is the state. Each doctor is a node and writes only their part. The referral rules
are the edges. The filing cabinet keeps the chart after every visit (the checkpointer). "Wait for the signed consent
form" is an interrupt. The next doctor picks up the chart and does not start the visit again.

## Picture
**Without LangGraph:** the loop does everything in one place, and its state is not saved.
```mermaid
flowchart LR
  C[Customer] --> L["while loop: model, tool, model"]
  L --> P["Payment inside the loop"]
  L -. "chat and calls: local variables" .-> X["lost on a crash"]
  P --> D["Money moved, no person looked"]
```

**With LangGraph:** the same steps are nodes, the pause is saved, and a person decides in between.
```mermaid
flowchart LR
  C[Customer] --> A["agent node: model call"]
  A -->|"lookup"| T["tools node"] --> A
  A -->|"refund"| I["approval node: interrupt()"]
  I -->|"saved pause, then resume"| P["pay node"] --> A
  A -->|"no tool calls"| E["END"]
  K[("checkpointer")] -.- I
```

## Already know this?
Part 4 (lessons 19 to 23) builds the support desk as a graph, adds MCP tools and grades whole runs. If you can answer
all five, **skip to lesson 23** (the Part's checkpoint):

1. Can you name what a graph's state holds, and why a node returns only the part it changed?
2. Can you say what a checkpointer saves, and what a thread id identifies?
3. Can you explain why code before `interrupt()` must be safe to run twice?
4. Can you name what an MCP server offers a client: tools, resources and prompts (lesson 21)?
5. Can you say why a whole run (its steps and tool calls) is graded, not only its final answer (lesson 22)?

## Step by step
**Step 1 · the desk (setup, no model call).** The support desk is a small file, `data/19/desk.py`, that the cell below
writes. It holds the fictional orders (made up for this course, as in lessons 06 to 14), the refund ledger, the two
tools, and the declarations the model reads: the name, what the tool does and its arguments. `refund_order` pays
through `LEDGER`, a key per order (lesson 14's idempotency key), so asking for the same refund twice pays once. The
rest of this lesson imports the desk, so its cells stay about the loop and the graph.

In [2]:
from pathlib import Path

Path("../data/19").mkdir(parents=True, exist_ok=True)  # this lesson's run-time folder

The desk file, written by the cell below. Read it once: it is the setup for the rest of the lesson.

In [3]:
%%writefile ../data/19/desk.py
"""The support desk of lesson 19: fictional orders, the refund ledger, two tools and their declarations.
No model call here. The rows are made up for this course."""
from google.genai import types
from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer

ORDERS = {
    "1093": {"status": "in transit", "days_late": 3, "total": 45.00},
    "2207": {"status": "delivered", "days_late": 0, "total": 84.00},
}
LEDGER = {}  # idempotency key -> the amount paid: one refund per order
PAID = []  # the money that has moved, one row per charge
QUESTION = "Order 2207 arrived with a torn box. Please refund the 84.00 USD I paid."


def lookup_order(order_id: str) -> dict:
    """Return the status of one order, or an error the model can read."""
    return ORDERS.get(order_id, {"error": f"no order with id {order_id}"})


def refund_order(order_id: str, amount_usd: float) -> dict:
    key = f"refund-{order_id}"  # the key: one refund per order
    if key in LEDGER:
        return {"status": "already paid", "amount_usd": LEDGER[key]}
    PAID.append((order_id, amount_usd))  # the money moves here
    LEDGER[key] = amount_usd
    return {"status": "refunded", "amount_usd": amount_usd}


TOOL_FUNCS = {"lookup_order": lookup_order, "refund_order": refund_order}  # name -> the Python function


def run_call(call):
    """Run one function request from the model; its result is the next turn's part."""
    return types.Part.from_function_response(name=call.name, response=TOOL_FUNCS[call.name](**call.args))


def calls_in(turn) -> list:  # the function requests in one model turn
    return [p.function_call for p in turn.parts if p.function_call]


def fallback_turn(history: list):  # offline only, when the model cell was skipped: fixed text, not model output
    if history[-1].parts[0].function_response:
        return types.Content(role="model", parts=[types.Part(text="(offline fallback, not model output) The refund is with the desk.")])
    refund = types.Part.from_function_call(name="refund_order", args={"order_id": "2207", "amount_usd": 84.0})
    return types.Content(role="model", parts=[refund])


def decl(name, description, properties, required):  # one tool declaration: name, what it does, its arguments
    return {"name": name, "description": description,
            "parameters_json_schema": {"type": "object", "properties": properties, "required": required}}


CONFIG = types.GenerateContentConfig(  # the system instruction and both tools, sent with every model call
    system_instruction="You are the Brightside Outfitters support desk. Use the tools for orders and refunds. "
                       "Tell the customer the outcome that the tool result reports.",
    tools=[{"function_declarations": [
        decl("lookup_order", "Look up the status of one order by its number.", {"order_id": {"type": "string"}}, ["order_id"]),
        decl("refund_order", "Refund an order. A person must approve it before any money moves.",
             {"order_id": {"type": "string"}, "amount_usd": {"type": "number"}}, ["order_id", "amount_usd"])]}],
    temperature=0, max_output_tokens=1024)  # the same settings as lesson 02

# the SDK classes a saved run may hold: without this list LangGraph warns on every load
SERDE = JsonPlusSerializer(allowed_msgpack_modules=[types.Content, types.Part, types.FunctionCall, types.FunctionResponse])

Overwriting ../data/19/desk.py


The desk is ready. The next cell imports it and the SDK's message types.

In [4]:
import sys; sys.path[:0] = ["..", "../data/19"]  # the course helpers, and the desk file written above
from google.genai import types

from desk import CONFIG, LEDGER, PAID, QUESTION, SERDE, calls_in, fallback_turn, refund_order, run_call

**Step 2 · the model (setup).** The key is read from `.env`, never typed here. The model is called through the official
Google SDK, the same way as lessons 06 and 10. `llm_replay` is a course helper: it saves each model call once, and
while a recording replays it uses a placeholder and sends no key. The model and its settings match lesson 02.

In [5]:
import llm_replay  # course helper: records and replays model calls (not part of the SDK)
llm_replay.start("19_why-langgraph")

from google import genai  # the official Google SDK
from tut_env import env  # course helper: reads a setting from the .env file (not part of the SDK)

MODEL = "gemini-3.1-flash-lite"  # the course's default model, the same as lessons 02 to 06
client = genai.Client(  # in your own project keep vertexai=True and pass your own key
    vertexai=True, api_key="replay" if llm_replay.is_replaying() else env("GOOGLE_CLOUD_API_KEY"))

LLM mode: replay (19_why-langgraph.yaml)


**Step 3 · one model call (the only place a call is made).** `ask_model` sends the turns so far and returns the
model's turn. A turn's function calls are its parts with a `function_call`. `CALLS` counts the real calls: this
notebook makes four, and the limit is five.

In [6]:
CALLS = {"n": 0}  # real model calls made in this kernel


def ask_model(history: list):
    if "client" not in globals():  # the model cell was skipped: the offline fallback answers instead
        return fallback_turn(history)
    CALLS["n"] += 1
    return client.models.generate_content(model=MODEL, contents=history, config=CONFIG).candidates[0].content

**Step 4 · the loop without a graph (2 model calls).** This is lesson 06's loop with the caps left out, so the shape
shows. The state is `history`, a local variable. The refund runs inside the loop, as soon as the model asks for it.

In [7]:
def loop_without_graph(question: str, max_steps: int = 6) -> str:
    history = [types.Content(role="user", parts=[types.Part(text=question)])]
    for _ in range(max_steps):
        turn = ask_model(history)  # one model call per pass
        history.append(turn)
        calls = calls_in(turn)
        if not calls:
            return turn.parts[0].text  # the answer ends the loop
        history.append(types.Content(role="user", parts=[run_call(c) for c in calls]))  # refund_order pays at once
    return "stopped: step cap reached"

**Run the loop** (2 model calls): the model asks for the refund, the loop pays, and the model answers.

In [8]:
PAID.clear(); LEDGER.clear()  # the loop starts with no money moved
answer = loop_without_graph(QUESTION)
print("answer:", answer, "\nmoney moved:", PAID, "· model calls so far:", CALLS["n"])

answer: I have processed your refund of 84.00 USD for order 2207. I'm sorry to hear that your package arrived in poor condition. 
money moved: [('2207', 84)] · model calls so far: 2


**Offline fallback** (runs only if the model cell above was skipped; the reply is typed in, not model output).

In [9]:
if "answer" not in globals():
    PAID.clear(); LEDGER.clear(); answer = loop_without_graph(QUESTION)
    print("answer:", answer, "\nmoney moved:", PAID, "· model calls so far:", CALLS["n"])

The loop finished, and the money moved before anyone decided. If the process had died after the payment, the
history would be gone, and the only trace would be the payment. Restarting the loop would start from the first
message and ask the model again. A pause would need your own store and resume code: lesson 14 wrote that for one
refund.

**Step 5 · the graph's state and nodes (no model call).** The state is a typed record. `history` has a reducer
(`operator.add`): a node's new turns are added to the list, and the old ones are kept. Each node returns only its
changes. `interrupt()` in the approval node pauses the run.

In [10]:
import operator
from typing import Annotated, TypedDict

from langgraph.checkpoint.sqlite import SqliteSaver  # the saver: writes each step to a SQLite file
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt  # interrupt pauses a run; Command resumes it with a value


class SupportState(TypedDict):
    history: Annotated[list, operator.add]  # the reducer: a node's new turns are appended, the old ones are kept
    refund: dict  # the refund request waiting for a decision
    decision: str  # "approve" or "reject", given by a person


def agent(state: SupportState) -> dict:
    return {"history": [ask_model(state["history"])]}


def tools(state: SupportState) -> dict:  # runs the lookups the model asked for
    return {"history": [types.Content(role="user", parts=[run_call(c) for c in calls_in(state["history"][-1])])]}


def approval(state: SupportState) -> dict:
    call = next(c for c in calls_in(state["history"][-1]) if c.name == "refund_order")
    decision = interrupt({"question": "Approve this refund?", **call.args})  # the run pauses here and is saved
    return {"refund": dict(call.args), "decision": decision}

**The payment node and the router (no model call).** The payment runs only after a person decides. `FLAKY` lets
Step 8 simulate a timeout. The router is the conditional edge: it reads the last turn and picks the next node.

In [11]:
FLAKY = {}  # "next": True makes the next payment time out before it charges (Step 8)


def pay(state: SupportState) -> dict:
    if FLAKY.pop("next", False):
        raise TimeoutError("payment service timed out before it charged")
    result = refund_order(**state["refund"]) if state["decision"] == "approve" else {"status": "rejected by a person"}
    return {"history": [types.Content(role="user", parts=[types.Part.from_function_response(name="refund_order", response=result)])]}


def route_after_agent(state: SupportState) -> str:
    calls = calls_in(state["history"][-1])
    return "approval" if any(c.name == "refund_order" for c in calls) else "tools" if calls else END

**Step 6 · compile with a checkpointer and draw the edges (no model call).** `build_app` wires the nodes and edges and
attaches the saver. The printed edges come from the compiled graph, so they are the real structure. The saver's
serializer (in the desk file) lists the SDK classes a saved run may hold, so it loads without a warning.

In [12]:
import sqlite3


def build_app(saver):
    g = StateGraph(SupportState)
    for name, fn in [("agent", agent), ("tools", tools), ("approval", approval), ("pay", pay)]:
        g.add_node(name, fn)
    g.add_edge(START, "agent")
    g.add_conditional_edges("agent", route_after_agent, {"approval": "approval", "tools": "tools", END: END})
    g.add_edge("tools", "agent").add_edge("approval", "pay").add_edge("pay", "agent")
    return g.compile(checkpointer=saver)


def open_app():  # a new connection and a new app object on the same file: what a restart looks like
    return build_app(SqliteSaver(sqlite3.connect(DB, check_same_thread=False), serde=SERDE))


DB = Path("../data/19/support_graph.db"); DB.unlink(missing_ok=True)  # a clean run: no saved runs from an earlier try
app = open_app()
CFG = {"configurable": {"thread_id": "desk-2207"}}  # one thread id = one saved run
print(sorted((e.source, e.target, e.conditional) for e in app.get_graph().edges))

[('__start__', 'agent', False), ('agent', '__end__', True), ('agent', 'approval', True), ('agent', 'tools', True), ('approval', 'pay', False), ('pay', 'agent', False), ('tools', 'agent', False)]


**Step 7 · run until the pause (1 model call).** The first model call asks for the refund. The approval node then
pauses the run, and the checkpointer saves it under the thread id. No money has moved.

In [13]:
PAID.clear(); LEDGER.clear()  # the graph run starts with no money moved, so its counts are its own
result = app.invoke({"history": [types.Content(role="user", parts=[types.Part(text=QUESTION)])]}, CFG)
print("paused for:", [i.value for i in result["__interrupt__"]], "\nwaiting at node:", app.get_state(CFG).next, "· money moved:", PAID, "· model calls:", CALLS["n"])

paused for: [{'question': 'Approve this refund?', 'amount_usd': 84, 'order_id': '2207'}] 
waiting at node: ('approval',) · money moved: [] · model calls: 3


**Offline fallback** (runs only if the model cell above was skipped; the reply is typed in, not model output).

In [14]:
if "result" not in globals():
    PAID.clear(); LEDGER.clear(); result = app.invoke({"history": [types.Content(role="user", parts=[types.Part(text=QUESTION)])]}, CFG)
    print("paused for:", [i.value for i in result["__interrupt__"]], "\nwaiting at node:", app.get_state(CFG).next)

**Step 8 · a restart, then a crash in the payment (no model call).** A new connection and a new app object stand in
for a restart: the only link to the run is the saved state and its thread id. A person approves. The payment times
out before it charges, so the run stops in the pay node.

In [15]:
app = open_app()  # a new connection and a new app object, as after a restart
FLAKY["next"] = True  # the payment service will time out once, as the next payment does
try:
    app.invoke(Command(resume="approve"), CFG)  # the person approves; the value becomes interrupt()'s result
except TimeoutError as e:
    print("crash in the payment node:", e)
print("saved and waiting at:", app.get_state(CFG).next, "· money moved:", PAID, "· model calls:", CALLS["n"])

crash in the payment node: payment service timed out before it charged
saved and waiting at: ('pay',) · money moved: [] · model calls: 3


**Step 9 · resume the saved run (1 model call, the answer).** `invoke(None, ...)` continues from the last saved step.
The approval is not asked for again, and the model is not asked for the refund again: only the pay node runs, then
the model writes the answer. `llm_replay.stop()` saves the recording after this, the last model call of this notebook.

In [16]:
final = app.invoke(None, CFG)
print("answer:", final["history"][-1].parts[0].text, "\nmoney moved:", PAID, "· model calls in total:", CALLS["n"], "(limit 5)")
llm_replay.stop()  # the last model call of this notebook

answer: I have processed your refund of 84.00 USD for order 2207. I'm sorry to hear that your package arrived in poor condition. 
money moved: [('2207', 84)] · model calls in total: 4 (limit 5)


**Offline fallback** (runs only if the model cell above was skipped; the reply is typed in, not model output).

In [17]:
if "final" not in globals():
    final = app.invoke(None, CFG)
    print("answer:", final["history"][-1].parts[0].text)

**Step 10 · the repeat is harmless (no model call).** If the pay node runs again after a crash, the key from Step 1
returns the first result and moves no money (lesson 14 explains the key).

In [18]:
print(refund_order(order_id="2207", amount_usd=84.0), "· money moved:", PAID)

{'status': 'already paid', 'amount_usd': 84} · money moved: [('2207', 84)]


## How it simplifies the architecture
The same refund, counted from the two runs above. The model client is the same in both.

| Moving part | Hand loop (lesson 06 with a refund) | LangGraph (this lesson) |
|---|---|---|
| Where the state lives | local variables, lost with the process | a typed state, saved after each node |
| A pause for a person | your own store, a status column and resume code (lesson 14) | `interrupt()` and `Command(resume=...)` |
| The order of steps | `if` and `while` code you read top to bottom | nodes and edges, drawn by `get_graph()` |
| A restart | starts again from the first message, and the model is asked again | resumes at the last saved step |
| The payment | runs inside the loop, before any person looks | runs after the approval node, once per key |

The graph adds a saver and a framework to learn. It removes the glue that a pause, a restart and a resume would
otherwise need in your code.

## At work
- LangGraph is named in about one agent-role listing in five in the job sample (22.6%). LangChain is named more often
  (34.5%). Teams choose between it, a vendor SDK and a hand loop → lesson 20 compares them on one task.
- An approval is a saved pause, not a waiting thread. LangGraph's `interrupt()` saves the run and resumes it later.
  The approval queue sits outside the worker → lesson 14 (the gate) and lesson 35 (durable runs with a worker kill).
- On resume, LangGraph re-runs the node that paused from its first line. Side effects before `interrupt()` must be
  safe to repeat. Keys do this, not the graph → lesson 32 (idempotency and retries).

## Common mistakes
1. **A side effect before `interrupt()`.** The paused node runs again from its first line on resume. This demo uses
   an in-memory saver, which is lost when the process ends. Anything that must survive a restart needs a saver that
   writes to a file, as in Step 6.

In [19]:
from langgraph.checkpoint.memory import InMemorySaver  # in memory: lost when the process ends

RUNS = []


def charge_then_ask(state: SupportState) -> dict:
    RUNS.append("charged")  # a side effect before interrupt(): it runs again on resume
    return {"decision": interrupt("approve?")}


demo_graph = StateGraph(SupportState).add_node("charge_then_ask", charge_then_ask).add_edge(START, "charge_then_ask")
demo = demo_graph.add_edge("charge_then_ask", END).compile(checkpointer=InMemorySaver())
demo.invoke({}, {"configurable": {"thread_id": "demo"}})  # pauses
demo.invoke(Command(resume="approve"), {"configurable": {"thread_id": "demo"}})  # the node runs again from its first line
print("charges recorded:", RUNS)  # two, not one: move the charge after interrupt(), behind a key

charges recorded: ['charged', 'charged']


2. **No checkpointer.** Without one, the first call returns as if it had paused, but nothing is saved. The resume
   below is the failure: there is no saved state to resume from.

In [20]:
no_saver = demo_graph.compile()  # the same graph, with no checkpointer
no_saver.invoke({}, {"configurable": {"thread_id": "lost"}})  # returns with an interrupt, saves nothing
no_saver.invoke(Command(resume="approve"), {"configurable": {"thread_id": "lost"}})  # raises: nothing to resume

RuntimeError: Cannot use Command(resume=...) without checkpointer

The error says what is missing. Attach a checkpointer when you compile, and resume with the same thread id.

3. **Reusing a thread id by accident.** A second request with the same thread id continues the old run and its
   state. Give each request its own id, such as the order's id plus the day.

## Try it
A refund at or under a limit can go straight to `pay`, with no person; a larger one goes to `approval`. Write
`route_with_limit(calls, limit=50.0)`: return `"pay"` for a refund at or under the limit, `"approval"` for a larger
refund, `"tools"` for a lookup, and `END` when there are no calls. A refund in the same reply as a lookup is routed
first. Check four cases: a refund of 45.00, a refund of 84.00, a lookup, and no calls. The graph is not changed, and no
model call is needed.

In [21]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [22]:
def route_with_limit(calls: list[dict], limit: float = 50.0) -> str:
    for c in calls:
        if c["name"] == "refund_order":
            return "pay" if c["args"]["amount_usd"] <= limit else "approval"
    return "tools" if calls else END


print(route_with_limit([{"name": "refund_order", "args": {"order_id": "1093", "amount_usd": 45.0}}]))
print(route_with_limit([{"name": "refund_order", "args": {"order_id": "2207", "amount_usd": 84.0}}]))
print(route_with_limit([{"name": "lookup_order", "args": {"order_id": "1093"}}]))
print(route_with_limit([]))

pay
approval
tools
__end__


## What a graph does not solve
- **Model choices stay random.** The next run may pick another tool. The graph decides where a run goes after each
  reply; it does not make the reply repeatable.
- **A charge that succeeds and then crashes runs again on resume.** The key in Step 1 stops the second charge (lesson 32).
- **More moving parts.** A saver, a database file and a framework version to pin. Lesson 20 weighs this against a
  vendor SDK and the hand loop; lesson 35 covers a durable workflow engine for runs that last hours.
- **When you don't need it:** one model call, or a fixed sequence with no pause (lesson 11).

## Recap
- A loop keeps its state in local variables. A graph keeps it in a typed state that a checkpointer saves.
- Nodes return only their changes. Edges, and conditional edges that read the state, decide what runs next.
- `interrupt()` pauses a run and saves it. `Command(resume=...)` continues the same thread, after a restart.
- On resume the paused node runs again from its first line, so payments need an idempotency key.

**Interview:** "Why use a graph framework instead of a while loop for an agent?" Strong answer: "A loop keeps its state
in variables, so a crash or an approval wait loses it. A graph saves the state after each node with a checkpointer,
so a run can pause for a person, survive a restart and resume without asking the model again. The payment still needs
an idempotency key."